In [6]:
from openai import OpenAI
import os
import pandas as pd
from pynvml import *
import time

In [2]:
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

# GPU model name
name = nvmlDeviceGetName(handle)
name = name.replace("NVIDIA GeForce ","")

# Memory info
mem_info = nvmlDeviceGetMemoryInfo(handle)
total_vram_gb = mem_info.total / (1024**3)

# GPU model identifier in the perf tables
gpu_model = f"{name} {total_vram_gb:.0f}GB"

print(f"Testing performance on the following GPU: {gpu_model}")

Testing performance on the following GPU: RTX 5090 32GB


In [3]:
def get_vram_used():
    info = nvmlDeviceGetMemoryInfo(handle)
    return info.used / 10**9

In [4]:
initial_vram = get_vram_used()
print(f"Initial VRAM used: {initial_vram:.3f}GB")

Initial VRAM used: 1.127GB


### vllm config

https://docs.vllm.ai/en/latest/configuration/engine_args/

--language-model-only, --no-language-model-only¶

If True, disables all multimodal inputs by setting all modality limits to 0. Equivalent to setting --limit-mm-per-prompt to 0 for every modality.
Default: False

--max-model-len¶

Model context length (prompt and output). If unspecified, will be automatically derived from the model config.

When passing via --max-model-len, supports k/m/g/K/M/G in human-readable format. Examples:

1k -> 1000
1K -> 1024
25.6k -> 25,600
-1 or 'auto' -> Automatically choose the maximum model length that fits in GPU memory. This will use the model's maximum context length if it fits, otherwise it will find the largest length that can be accommodated.
Parse human-readable integers like '1k', '2M', etc. Including decimal values with decimal multipliers. Also accepts -1 or 'auto' as a special value for auto-detection.


Examples:
- '1k' -> 1,000
- '1K' -> 1,024
- '25.6k' -> 25,600
- '-1' or 'auto' -> -1 (special value for auto-detection)

--quantization, -q¶

Method used to quantize the weights. If None, we first check the quantization_config attribute in the model config file. If that is None, we assume the model weights are not quantized and use dtype to determine the data type of the weights.

--kv-cache-dtype¶

Possible choices: auto, bfloat16, float16, fp8, fp8_ds_mla, fp8_e4m3, fp8_e5m2, fp8_inc, fp8_per_token_head, int8_per_token_head
Data type for kv cache storage. If "auto", will use model data type. CUDA 11.8+ supports fp8 (=fp8_e4m3) and fp8_e5m2. ROCm (AMD GPU) supports fp8 (=fp8_e4m3). Intel Gaudi (HPU) supports fp8 (using fp8_inc). Some models (namely DeepSeekV3.2) default to fp8, set to bfloat16 to use bfloat16 instead, this is an invalid option for models that do not default to fp8.
Default: auto

--kv-offloading-size¶

Size of the KV cache offloading buffer in GiB. When TP > 1, this is the total buffer size summed across all TP ranks. By default, this is set to None, which means no KV offloading is enabled. When set, vLLM will enable KV cache offloading to CPU using the kv_offloading_backend.

--kv-offloading-backend¶

Possible choices: lmcache, native
The backend to use for KV cache offloading. Supported backends include 'native' (vLLM native CPU offloading), 'lmcache'. KV offloading is only activated when kv_offloading_size is set.
Default: native

--enable-prefix-caching, --no-enable-prefix-caching¶

Whether to enable prefix caching.

--max-num-seqs¶

Maximum number of sequences to be processed in a single iteration.

The default value here is mainly for convenience when testing. In real usage, this should be set in EngineArgs.create_engine_config.

--moe-backend¶

Possible choices: aiter, auto, cutlass, deep_gemm, flashinfer_cutedsl, flashinfer_cutlass, flashinfer_trtllm, marlin, triton
Backend for MoE expert computation kernels. Available options:

"auto": Automatically select the best backend based on model and hardware
"triton": Use Triton-based fused MoE kernels
"deep_gemm": Use DeepGEMM kernels (FP8 block-quantized only)
"cutlass": Use vLLM CUTLASS kernels
"flashinfer_trtllm": Use FlashInfer with TRTLLM-GEN kernels
"flashinfer_cutlass": Use FlashInfer with CUTLASS kernels
"flashinfer_cutedsl": Use FlashInfer with CuteDSL kernels (FP4 only)
"marlin": Use Marlin kernels (weight-only quantization)
"aiter": Use AMD AITer kernels (ROCm only)
Default: auto

--speculative-config, -sc¶

Speculative decoding configuration.

API docs: https://docs.vllm.ai/en/latest/api/vllm/config/#vllm.config.SpeculativeConfig

Should either be a valid JSON string or JSON keys passed individually.

--attention-config, -ac¶

Attention configuration.

API docs: https://docs.vllm.ai/en/latest/api/vllm/config/#vllm.config.AttentionConfig

Should either be a valid JSON string or JSON keys passed individually.

Default: AttentionConfig(backend=None, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False)

--reasoning-config¶

The configurations for reasoning model.

API docs: https://docs.vllm.ai/en/latest/api/vllm/config/#vllm.config.ReasoningConfig

Should either be a valid JSON string or JSON keys passed individually.

--optimization-level¶

The optimization level. These levels trade startup time cost for performance, with -O0 having the best startup time and -O3 having the best performance. -O2 is used by default. See OptimizationLevel for full description.
Default: 2

--performance-mode¶

Possible choices: balanced, interactivity, throughput
Performance mode for runtime behavior, 'balanced' is the default. 'interactivity' favors low end-to-end per-request latency at small batch sizes (fine-grained CUDA graphs, latency-oriented kernels). 'throughput' favors aggregate tokens/sec at high concurrency (larger CUDA graphs, more aggressive batching, throughput-oriented kernels).
Default: balanced

> docker pull vllm/vllm-openai:v0.19.0-cu130-ubuntu2404

> vi dockerfile

```
FROM vllm/vllm-openai:v0.19.0-cu130-ubuntu2404

# Upgrade transformers to latest
RUN pip install transformers==5.5.0
```

> docker build -t vllm-openai:v0.19.0-cu130-transformers5.5.0 .

> mkdir $WORDSLAB_MODELS/vllm 

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm-openai:v0.19.0-cu130-transformers5.5.0 nvidia/Gemma-4-31B-IT-NVFP4 --quantization modelopt --language-model-only --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.92 --host 0.0.0.0 --port 8000
```

nvidia/Gemma-4-31B-IT-NVFP4 => not enough GPU memory available to serve even a single token.

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm/vllm-openai:gemma4-cu130 RedHatAI/gemma-4-26B-A4B-it-NVFP4 --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.92 --host 0.0.0.0 --port 8000
```

RedHatAI/gemma-4-26B-A4B-it-NVFP4 => KeyError: 'layers.0.experts.0.down_proj.input_global_scale'

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm/vllm-openai:v0.19.0-cu130-ubuntu2404 apolo13x/Qwen3.5-27B-NVFP4 --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder --speculative-config '{"method":"mtp","num_speculative_tokens":2}' --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.94 --host 0.0.0.0 --port 8000
```


apolo13x/Qwen3.5-27B-NVFP4 - with MTP => Auto-fit max_model_len: reduced from 262144 to 71200 to fit in available GPU memory (5.27 GiB available for KV cache)

apolo13x/Qwen3.5-27B-NVFP4 - without MTP => Auto-fit max_model_len: reduced from 262144 to 92512 to fit in available GPU memory (5.95 GiB available for KV cache

cyankiwi/Qwen3.5-27B-AWQ-4bit => Auto-fit max_model_len: reduced from 262144 to 92512 to fit in available GPU memory (5.96 GiB available for KV cache)

LilaRest/gemma-4-31B-it-NVFP4-turbo => Auto-fit max_model_len: reduced from 262144 to 162336 to fit in available GPU memory (9.72 GiB available for KV cache)

Sehyo/Qwen3.5-35B-A3B-NVFP4

In [20]:
client = OpenAI(base_url="http://127.0.0.1:8000/v1",api_key="")

In [83]:
model = "LilaRest/gemma-4-31B-it-NVFP4-turbo"
max_context_length = 162336

In [84]:
def compute_prompt_tokens(prompt):
    cc = client.chat.completions.create(model=model, messages = [{'role': 'user', 'content': prompt}], max_tokens = 1)
    return cc.usage.prompt_tokens

In [85]:
def measure_perfs(prompt_repeat):
    start = time.perf_counter()
    stream = client.chat.completions.create(model=model, messages = [{'role': 'user', 'content': (base_text*prompt_repeat)}], max_tokens = 100, stream=True, stream_options={"include_usage": True})
    first_token_time = None
    for chunk in stream:
        if first_token_time is None:
            first_token_time = time.perf_counter()
            ttf = first_token_time - start
        # capture usage if present (usually only in final chunk)
        if hasattr(chunk, "usage") and chunk.usage:
            usage = chunk.usage
    end = time.perf_counter()
    total = end - start
    return usage.prompt_tokens,usage.prompt_tokens/ttf,usage.completion_tokens,usage.completion_tokens/total
#    return response.prompt_eval_count,response.prompt_eval_count/(response.prompt_eval_duration/10**9), response.eval_count,response.eval_count/(response.eval_duration/10**9)

In [86]:
context_sizes = [1024*p for p in range(1,8)] + [8192*p for p in range(1,17)] + [16384*p for p in range(9,17)] + [32768*p for p in range(9,17)] + [65536*p for p in range(9,17)]

In [87]:
ctx1k = context_sizes[0]
ctx2k = context_sizes[1]
ctx4k = context_sizes[3]
ctx8k = context_sizes[7]
ctx16k = context_sizes[8]
ctx32k = context_sizes[10]
ctx64k = context_sizes[14]
ctx128k = context_sizes[22]
ctx192k = context_sizes[26]
ctx256k = context_sizes[30]
ctx384k = context_sizes[34]
ctx512k = context_sizes[38]
ctx1M = context_sizes[46]

In [88]:
print(",".join([f"{int(context_sizes[ctx_index]/1024)}k" for ctx_index in range(7,35)]))
print(",".join([f"{context_sizes[ctx_index]}" for ctx_index in range(7,35)]))

8k,16k,24k,32k,40k,48k,56k,64k,72k,80k,88k,96k,104k,112k,120k,128k,144k,160k,176k,192k,208k,224k,240k,256k,288k,320k,352k,384k
8192,16384,24576,32768,40960,49152,57344,65536,73728,81920,90112,98304,106496,114688,122880,131072,147456,163840,180224,196608,212992,229376,245760,262144,294912,327680,360448,393216


In [89]:
base_text = '\nAbsolutely! Let’s unpack this carefully and thoroughly. The concept of a **token** comes up most often in natural language processing (NLP), programming, and blockchain/crypto, but I’ll focus first on the context most relevant to AI models like me, and then touch briefly on other uses for clarity.\n\n---\n\n## **1. Tokens in AI and NLP (Language Models)**\n\nIn the context of language models like GPT:\n\n**A token is essentially a piece of text that the model treats as a unit for processing.**\n\nThink of it as a “chunk” of text, which could be:\n\n* A word (“apple” → 1 token)\n* Part of a word (“unbelievable” → might be split into “un”, “believ”, “able” → 3 tokens)\n* A single character or punctuation mark ("," or "!" → 1 token)\n* Even whitespace in some systems.\n\nLanguage models **do not process text character by character or word by word** in the strict sense; they process these tokens. This is because using tokens allows the model to handle text efficiently while capturing meaningful patterns.\n\n### **Why not just words?**\n\nWords are tricky because:\n\n* Some words are very long and rare.\n* Some languages (like Chinese, Japanese) don’t use spaces to separate words.\n* Using subword tokens balances between vocabulary size and representation power.\n\nBy using tokens, a model can efficiently represent language while keeping the total vocabulary manageable.\n\n---\n\n### **Tokenization: How text becomes tokens**\n\nThe process of breaking text into tokens is called **tokenization**. There are different tokenization methods:\n\n1. **Whitespace tokenization**\n\n   * Splits text at spaces.\n   * `"I love pizza!"` → `["I", "love", "pizza!"]`\n   * Simple, but can miss nuances like punctuation.\n\n2. **Subword tokenization (most modern models use this)**\n\n   * Breaks rare words into smaller pieces.\n   * `"unbelievable"` → `["un", "believ", "able"]`\n   * Helps the model understand new words it hasn\'t seen during training.\n\n3. **Character-level tokenization**\n\n   * Treats each character as a token.\n   * `"cat"` → `["c", "a", "t"]`\n   * Rarely used alone because sequences become very long.\n\n4. **Byte Pair Encoding (BPE)**\n\n   * A popular method used in GPT models.\n   * Starts with characters and iteratively merges common sequences to form subwords.\n   * Efficiently handles rare words while keeping the vocabulary compact.\n\n---\n\n### **Example of token counts**\n\nLet’s take the sentence:\n\n```\nHello, world! I love programming.\n```\n\nA GPT-style tokenizer might break it into tokens like:\n\n```\n["Hello", ",", " world", "!", " I", " love", " program", "ming", "."]\n```\n\nHere, it’s **9 tokens**. Notice:\n\n* Punctuation is separate.\n* Spaces before words are included as part of the token in some tokenizers.\n* Words like “programming” are split.\n\n---\n\n### **Why tokens matter in language models**\n\n1. **Input length**\n\n   * Models have a limit on the number of tokens they can process at once (e.g., GPT-4’s 8,192 or 32k token limit).\n   * Longer text → more tokens → may exceed the model’s limit.\n\n2. **Cost & computation**\n\n   * In APIs like OpenAI’s, you pay per token.\n   * More tokens → more computation → higher cost.\n\n3. **Context & memory**\n\n   * Models can only “remember” information within a token limit.\n   * Efficient tokenization allows models to fit more meaningful text into memory.\n\n---\n\n## **2. Tokens in Programming**\n\nA token in programming is a **syntactic unit of code**:\n\n* Keywords (`if`, `for`, `return`)\n* Identifiers (`myVariable`, `functionName`)\n* Operators (`+`, `-`, `=`)\n* Literals (`42`, `"hello"`)\n* Punctuation (`;`, `{`, `}`)\n\n**Example:**\n\n```python\nx = 42 + 7\n```\n\nTokenization would produce:\n\n```\n["x", "=", "42", "+", "7"]\n```\n\nThis is similar in principle: breaking a stream of characters into meaningful chunks the compiler or interpreter can understand.\n\n---\n\n## **3. Tokens in Blockchain / Crypto**\n\nHere, a token is a **digital asset** issued on a blockchain, representing something of value:\n\n* **Utility tokens** – give access to a service (e.g., Binance Coin for exchange fees).\n* **Security tokens** – represent ownership or investment.\n* **NFTs (non-fungible tokens)** – unique digital items (art, collectibles).\n\nThis is different from NLP, but the common theme is that a token is a **unit of something meaningful**, whether text, code, or digital value.\n\n---\n\n## **Key Takeaways**\n\n* **In AI/NLP:** Token = smallest meaningful piece of text for a model to process.\n* **In programming:** Token = smallest meaningful element of code.\n* **In crypto:** Token = a digital representation of value or rights.\n\n**Analogy:** Tokens are like LEGO blocks. Each block may be a full piece (word) or a part of a bigger structure (subword). You can build complex things (sentences, programs, ecosystems) from these basic building blocks.\n\n---\n\nIf you want, I can make a **visual diagram showing how a sentence gets broken into tokens and why that matters for GPT**, which makes this concept much easier to grasp.\n\nDo you want me to do that?\n'

In [90]:
# Load the model and tokenize the test text
base_tokens = compute_prompt_tokens(base_text)   

# Test at all prompt sizes until context_length - 100 (output tokens)
previous_prompt_tokens = -1
for prompt_size in context_sizes:
    prompt_repeat = int(prompt_size/base_tokens)
    prompt_tokens = prompt_repeat * base_tokens
    if prompt_tokens > (max_context_length - 100):
        break
    if prompt_tokens != previous_prompt_tokens:
        previous_prompt_tokens = prompt_tokens
        start_time = time.time()
        real_prompt_tokens,prompt_tokens_per_sec, output_tokens,output_tokens_per_sec = measure_perfs(prompt_repeat)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"{gpu_model},{model},{real_prompt_tokens},{int(prompt_tokens_per_sec)},{output_tokens_per_sec:.2f}")
        if elapsed_time > 60:
            break

RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,13,51,45.21
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,1271,22627,48.66
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,2530,3461,36.39
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,3789,27752,46.19
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,5048,36545,45.75
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,6307,41141,45.11
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,7566,46422,44.52
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,15120,14757,31.30
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,23933,5664,15.45
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,31487,5363,12.28
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,40300,8451,14.16
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,47854,23128,22.84
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,56667,10451,12.89
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,64221,24277,19.98
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,73034,11999,11

ReadTimeout: timed out

NVFP4 with MTP

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm/vllm-openai:v0.19.0-cu130-ubuntu2404 apolo13x/Qwen3.5-27B-NVFP4 --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder --speculative-config '{"method":"mtp","num_speculative_tokens":2}' --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.94 --host 0.0.0.0 --port 8000
```

```
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,10,84,36.44
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,1271,9141,34.60
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,2533,3200,27.07
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,3795,13434,30.70
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,5057,18152,29.80
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,6319,21208,28.21
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,7581,23971,27.48
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,15153,16060,20.22
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,23987,18471,16.39
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,31559,23974,14.66
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,40393,24265,12.57
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,47965,28997,11.59
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,56799,26451,10.08
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,64371,30683,9.45
```

NVFP4 without MTP

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm/vllm-openai:v0.19.0-cu130-ubuntu2404 apolo13x/Qwen3.5-27B-NVFP4 --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.94 --host 0.0.0.0 --port 8000
```

```
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,10,41,49.75
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,1271,10303,52.75
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,2533,6691,46.56
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,3795,17343,50.03
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,5057,21464,49.68
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,6319,35338,50.88
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,7581,34820,49.85
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,15153,17736,37.44
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,23987,22489,34.29
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,31559,27689,33.13
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,40393,28122,29.88
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,47965,33312,29.60
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,56799,32454,26.87
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,64371,36683,26.66
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,73205,34704,24.06
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,80777,38950,24.10
RTX 5090 32GB,apolo13x/Qwen3.5-27B-NVFP4,88349,40990,23.53
```

AWQ-4bits

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm/vllm-openai:v0.19.0-cu130-ubuntu2404 cyankiwi/Qwen3.5-27B-AWQ-4bit --reasoning-parser qwen3 --enable-auto-tool-choice --tool-call-parser qwen3_coder --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.94 --host 0.0.0.0 --port 8000
```

```
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,10,36,43.11
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,1271,8700,68.50
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,2533,4141,51.86
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,3795,9741,58.27
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,5057,10068,54.72
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,6319,13845,55.79
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,7581,19986,58.27
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,15153,7013,28.37
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,23987,9123,24.83
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,31559,12752,25.63
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,40393,13757,22.75
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,47965,17215,23.40
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,56799,17319,20.82
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,64371,20738,21.52
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,73205,20145,19.12
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,80777,23665,19.85
RTX 5090 32GB,cyankiwi/Qwen3.5-27B-AWQ-4bit,88349,25663,19.62
```

NVFP4 Turbo

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface -v $WORDSLAB_MODELS/vllm:/root/.cache/vllm --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000 --ipc=host --shm-size 16G vllm-openai:v0.19.0-cu130-transformers5.5.0 LilaRest/gemma-4-31B-it-NVFP4-turbo --trust-remote-code --quantization modelopt --max-num-seqs 1 --max-num-batched-tokens 8192 --kv-cache-dtype fp8 --max-model-len auto --enable-prefix-caching --gpu-memory-utilization 0.94 --host 0.0.0.0 --port 8000
```

```
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,13,51,45.21
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,1271,22627,48.66
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,2530,3461,36.39
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,3789,27752,46.19
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,5048,36545,45.75
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,6307,41141,45.11
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,7566,46422,44.52
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,15120,14757,31.30
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,23933,5664,15.45
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,31487,5363,12.28
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,40300,8451,14.16
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,47854,23128,22.84
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,56667,10451,12.89
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,64221,24277,19.98
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,73034,11999,11.80
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,80588,25081,17.79
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,88142,25184,16.88
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,96955,13739,10.50
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,104509,25609,15.23
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,113322,14647,9.76
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,120876,25877,13.88
RTX 5090 32GB,LilaRest/gemma-4-31B-it-NVFP4-turbo,129689,15340,9.09
```